In [2]:
pip install pandas numpy scikit-learn xgboost lightgbm catboost imbalanced-learn matplotlib seaborn ipywidgets joblib

  Using cached fqdn-1.5.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -------- ------------------------------ 22.0/101.7 MB 115.7 MB/s eta 0:00:01
   ----------------- --------------------- 45.6/101.7 MB 116.0 MB/s eta 0:00:01
   -------------------------- ------------ 69.2/101.7 MB 116.1 MB/s eta 0:00:01
   ----------------------------------- --- 92.8/101.7 MB 116.1 MB/s eta 0:00:01
   -------------------------------------  101.4/101.7 MB 115.6 MB/s eta 0:00:01
   ---------------------------------------- 101.7/101.7 MB 84.2 MB/s  0:00:01
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 74.3 MB/s  0:00:00
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   --------- ------------------------

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
import joblib
from sklearn.preprocessing import StandardScaler


In [5]:
data = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import RandomOverSampler

# Resampling using RandomOverSampler
ros = RandomOverSampler(random_state=42)
features = data.drop(['Diabetes_binary', 'Fruits', 'Veggies'], axis=1)  # Excluding fruits and veggies from the model
target = data['Diabetes_binary']
features_resampled, target_resampled = ros.fit_resample(features, target)

# Splitting the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    features_resampled, 
    target_resampled, 
    test_size=0.3, 
    random_state=42
)

# Defining and training models
log_reg = LogisticRegression(max_iter=1000)
random_forest = RandomForestClassifier(n_estimators=100)
gbm = GradientBoostingClassifier(n_estimators=100)
svm = SVC(kernel='linear')
neural_network = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000)

models = [log_reg, random_forest, neural_network]
model_names = ['Logistic Regression', 'Random Forest']

# Training and evaluating each model
for model, name in zip(models, model_names):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} - Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(f"{name} - Precision: {precision_score(y_test, y_pred, average='binary'):.2f}")
    print(f"{name} - Recall: {recall_score(y_test, y_pred, average='binary'):.2f}")
    print(f"{name} - F1 Score: {f1_score(y_test, y_pred, average='binary'):.2f}\n")


Logistic Regression - Accuracy: 0.75
Logistic Regression - Precision: 0.74
Logistic Regression - Recall: 0.76
Logistic Regression - F1 Score: 0.75



In [ ]:
# Assuming all other features are numerical or have been encoded as such
numerical_columns = [col for col in features.columns if features[col].dtype != 'object']

# Initialize and fit the StandardScaler
scaler = StandardScaler()
features[numerical_columns] = scaler.fit_transform(features[numerical_columns])

# Splitting the resampled data into train and test sets
X_train_resampled, X_test_resampled, y_train_resampled, y_test_resampled = train_test_split(
    features_resampled, target_resampled, test_size=0.3, random_state=42
)
# Define and train Random Forest model
random_forest_model = RandomForestClassifier(n_estimators=100)
random_forest_model.fit(X_train_resampled, y_train_resampled)

# Evaluate the model
y_pred_resampled = random_forest_model.predict(X_test_resampled)
print("Accuracy:", accuracy_score(y_test_resampled, y_pred_resampled))
print("Precision:", precision_score(y_test_resampled, y_pred_resampled, average='binary'))
print("Recall:", recall_score(y_test_resampled, y_pred_resampled, average='binary'))
print("F1 Score:", f1_score(y_test_resampled, y_pred_resampled, average='binary'))

# Save the model and scaler
joblib.dump(random_forest_model, 'random_forest_model_upsampled.joblib')
joblib.dump(scaler, 'scaler.joblib')


In [ ]:
# Load the trained model and scaler
random_forest_model = joblib.load('random_forest_model_upsampled.joblib')
scaler = joblib.load('scaler.joblib')

# Helper function for creating descriptive labels
def create_description(description):
    return widgets.Label(value=f'{description}')

# Define widgets with user-friendly descriptions
widgets_dict = {
    "HighBP": (create_description('Do you have high blood pressure?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "HighChol": (create_description('Do you have high cholesterol?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "CholCheck": (create_description('Have you checked your cholesterol in the last 5 years?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "BMI": (create_description('What is your Body Mass Index (BMI)?'), widgets.FloatSlider(value=25.0, min=10, max=50.0, step=0.1, description='')),
    "Smoker": (create_description('Have you smoked over 100 cigarettes in your lifetime?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "Stroke": (create_description('Have you ever had a stroke?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "HeartDiseaseorAttack": (create_description('Have you ever had heart disease or a heart attack?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "PhysActivity": (create_description('Do you regularly engage in physical activity instead of going to your job?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "HvyAlcoholConsump": (create_description('Do you consume alcohol heavily?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "AnyHealthcare": (create_description('Do you have any healthcare coverage?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "NoDocbcCost": (create_description('Have you ever needed to see a doctor but could not due to cost?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "GenHlth": (create_description('How would you rate your general health?'), widgets.Dropdown(options=[('Excellent', 1), ('Very Good', 2), ('Good', 3), ('Fair', 4), ('Poor', 5)], description='')),
    "MentHlth": (create_description('How many days have your mental health not been good in the past month?'), widgets.IntSlider(value=0, min=0, max=30, step=1, description='')),
    "PhysHlth": (create_description('How many days have your physical health not been good in the past month?'), widgets.IntSlider(value=0, min=0, max=30, step=1, description='')),
    "DiffWalk": (create_description('Do you have difficulty walking or climbing stairs?'), widgets.RadioButtons(options={"No": 0, "Yes": 1}, description='')),
    "Sex": (create_description('What is your gender?'), widgets.RadioButtons(options={"Female": 0, "Male": 1}, description='')),
    "Age": (create_description('What is your age?'), widgets.Dropdown(options=[('18-24', 1), ('25-29', 2), ('30-34', 3), ('35-39', 4), ('40-44', 5), ('45-49', 6), ('50-54', 7), ('55-59', 8), ('60-64', 9), ('65-69', 10), ('70-74', 11), ('75-79', 12), ('80+', 13)], description='')),
    "Education": (create_description('What is the highest level of education you have completed?'), widgets.Dropdown(options=[('Never attended school', 1), ('Elementary school', 2), ('Some high school', 3), ('High school graduate', 4), ('Some college', 5), ('College graduate', 6)], description='')),
    "Income": (create_description('What is your income category?'), widgets.Dropdown(options=[('$10,000 or less', 1), ('$10,000 to $15,000', 2), ('$15,000 to $20,000', 3), ('$20,000 to $25,000', 4), ('$25,000 to $35,000', 5), ('$35,000 to $50,000', 6), ('$50,000 to $75,000', 7), ('$75,000 or more', 8)], description=''))
}

# Button to make prediction
predict_btn = widgets.Button(description="Predict Diabetes Risk")

# Output widget to display prediction result and advice
output = widgets.Output()

def on_predict_btn_clicked(b):
    # Prepare the input for the model
    input_data = [widget.value for label, widget in widgets_dict.values()]
    input_data = np.array(input_data).reshape(1, -1)

    # Apply the same scaling to the input as was done to the training data
    input_data_scaled = scaler.transform(input_data)

    # Make prediction and calculate probabilities
    prediction = random_forest_model.predict(input_data_scaled)
    prediction_probabilities = random_forest_model.predict_proba(input_data_scaled)
    risk_probability = prediction_probabilities[0][1] * 100  # Probability of being at high risk

    # Display prediction, confidence, and detailed recommendations
    with output:
        output.clear_output()
        if prediction[0] == 1:
            print("Prediction: You are currently at high risk of diabetes.")
            print("Immediate actions are recommended:")
        else:
            print("Prediction: You do not currently have diabetes.")
            print(f"However, you are at {risk_probability:.2f}% risk of developing diabetes in the future.")
            print("To lower your risk, consider the following tips:")

        for importance, feature in sorted(zip(random_forest_model.feature_importances_, widgets_dict.keys()), key=lambda x: x[0], reverse=True)[:3]:
            print(f"- {feature}: {get_recommendation(feature, high_risk=prediction[0] == 1)}")

def get_recommendation(feature, high_risk):
    recommendations = {
        'HighBP': {
            True: "Monitor and manage your blood pressure through diet and medication.",
            False: "Maintaining a healthy blood pressure is vital. Regular check-ups are recommended."
        },
        'HighChol': {
            True: "Consider dietary changes and medication to manage cholesterol.",
            False: "Continue eating a diet low in fats and cholesterol to maintain good health."
        },
        'BMI': {
            True: "Consider a weight loss plan that includes diet and exercise to reduce your BMI.",
            False: "Maintaining a healthy BMI through balanced diet and exercise helps prevent diabetes."
        },
        'Smoker': {
            True: "Quitting smoking is essential for reducing health risks.",
            False: "If you smoke, quitting can significantly reduce your health risks."
        },
        'PhysActivity': {
            True: "Increasing physical activity can help reduce weight and improve insulin sensitivity.",
            False: "Regular physical activity is beneficial for maintaining healthy weight and reducing diabetes risk."
        }
    }
    return recommendations.get(feature, {}).get(high_risk, "Maintaining a balanced lifestyle is crucial.")

predict_btn.on_click(on_predict_btn_clicked)

# Display widgets
for label, widget in widgets_dict.values():
    display(label, widget)
display(predict_btn, output)
